# CocoGuard: 3-model reliability and explainability study (ResNet50, EfficientNetB0, MobileNetV2)
Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. Everything is saved to Google Drive.

Pipeline (follows the research plan):
1. Leakage-aware split (near-duplicate removal, frozen manifest) + leakage audit
2. Train the three models with the **same** split, preprocessing, augmentation, class weights and metrics
3. Baseline: macro-F1, per-class recall (bootstrap CIs), confusion matrices
4. Robustness: blur / dark / bright / JPEG at 3 severities, per-class vulnerability
5. Calibration (ECE, Brier, temperature scaling fitted on corrupted validation data) + selective prediction
6. Grad-CAM stability under corruption + cross-model explanation consistency
7. Idea C: does explanation instability help detect errors? (deployable test, see the cell for details)
8. Optional external test set

To use YOLO26-cls as one of the three models, replace a name in `MODELS` and reuse your earlier YOLO notebook for it.

In [ ]:
!pip install -q imagehash kagglehub grad-cam scikit-image

## 1. Settings

In [ ]:
import os, io, glob, json, random, shutil, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/cocoguard_3models_v2'
RES = f'{OUT}/results'; os.makedirs(f'{RES}/figures', exist_ok=True)

MODELS = ['resnet50', 'efficientnet_b0', 'mobilenet_v2']
EPOCHS, IMGSZ, BATCH, SEED, LR = 15, 224, 32, 42, 3e-4
TRAIN = True                 # keep True for the first run on the new split; False = reuse weights already saved in Drive
N_CAM, N_IDEA_C = 150, 300   # images used for Grad-CAM stability / idea C (lower these if Colab is slow)
EXT_DIR = None               # optional: folder with your own photos, one sub-folder per class (same names as the dataset classes)

random.seed(SEED); np.random.seed(SEED)
import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

## 2. Download, group near-duplicates, split 70/15/15, freeze
Near-duplicate photos (shifted / colour-adjusted copies) are clustered by perceptual hash, and each **whole cluster goes into a single split**, so a copy of a test photo can never sit in train.
The split is saved as a manifest in Drive, so every later run (and every model) uses exactly the same images.

In [ ]:
import kagglehub, imagehash
from PIL import Image
from tqdm.auto import tqdm
import torchvision.transforms as T
import scipy.sparse as sps
from scipy.sparse.csgraph import connected_components

path = kagglehub.dataset_download("nagabushan/coconut-tree-disease-dataset")
IMGEXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def has_imgs(d): return any(os.path.splitext(f)[1].lower() in IMGEXT for f in os.listdir(d))
def class_dirs(d): return [s for s in sorted(os.listdir(d)) if os.path.isdir(f'{d}/{s}') and has_imgs(f'{d}/{s}')]
ROOT = [dp for dp, _, _ in os.walk(path) if len(class_dirs(dp)) >= 2][0]
classes = class_dirs(ROOT); K = len(classes)
print('Dataset root:', ROOT); print({c: len(os.listdir(f'{ROOT}/{c}')) for c in classes})

POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def phash_bits(p): return np.packbits(imagehash.phash(Image.open(p).convert('RGB'), hash_size=16).hash.flatten())

GROUP_THR = 40                                   # bits out of 256: images closer than this count as the same photo
MAN  = f'{OUT}/split_manifest_group.json'
DATA = '/content/coconut_split_group'
allf = [(c, f) for c in classes for f in sorted(os.listdir(f'{ROOT}/{c}')) if os.path.splitext(f)[1].lower() in IMGEXT]

if os.path.exists(MAN):
    manifest = json.load(open(MAN)); print('Loaded frozen group split:', len(manifest), 'images')
else:
    Hh = np.stack([phash_bits(f'{ROOT}/{c}/{f}') for c, f in tqdm(allf, desc='hashing')])
    r_, c_ = [], []
    for i in range(len(Hh)):
        d = POP[Hh[i + 1:] ^ Hh[i]].sum(1); j = np.where(d <= GROUP_THR)[0] + i + 1
        r_ += [i] * len(j); c_ += j.tolist()
    A = sps.coo_matrix((np.ones(len(r_)), (r_, c_)), shape=(len(Hh), len(Hh)))
    ng, grp = connected_components(A, directed=False)
    cls_of = np.array([classes.index(c) for c, _ in allf]); gsize = np.bincount(grp)
    print(f'images: {len(Hh)} | groups: {ng} | images in multi-image groups: {int((gsize[grp] > 1).sum())} | largest group: {gsize.max()} | '
          f'groups mixing classes: {sum(len(set(cls_of[grp == g])) > 1 for g in range(ng))}')
    maj = np.array([np.bincount(cls_of[grp == g], minlength=K).argmax() for g in range(ng)])   # each group goes to ONE split, balanced per majority class
    rg = random.Random(SEED); split_of = {}
    for k in range(K):
        gs = [g for g in range(ng) if maj[g] == k]; rg.shuffle(gs); tot = sum(gsize[g] for g in gs); cum = 0
        for g in gs:
            split_of[g] = 'train' if cum < .70 * tot else ('val' if cum < .85 * tot else 'test'); cum += gsize[g]
    manifest = [[split_of[grp[i]], allf[i][0], allf[i][1]] for i in range(len(allf))]
    json.dump(manifest, open(MAN, 'w')); print('Group split frozen ->', MAN)

if not os.path.exists(f'{DATA}/.done'):
    shutil.rmtree(DATA, ignore_errors=True)
    for sp, c, f in tqdm(manifest, desc='building split folders'):
        os.makedirs(f'{DATA}/{sp}/{c}', exist_ok=True)
        T.Resize(256)(Image.open(f'{ROOT}/{c}/{f}').convert('RGB')).save(f'{DATA}/{sp}/{c}/{os.path.splitext(f)[0]}.png')
    open(f'{DATA}/.done', 'w').write('ok')
for sp in ['train', 'val', 'test']: print(sp.ljust(6), {c: len(os.listdir(f'{DATA}/{sp}/{c}')) for c in classes})

## 3. Leakage audit
For every val/test image: Hamming distance (out of 256 bits) to its **nearest training image**. Small numbers mean a near-copy sits in train.
Look at the closest pairs yourself. If they still show the same scene, raise `GROUP_THR` (try 50-60), delete the manifest in Drive, and rerun. Report this check in the paper.

In [ ]:
def files(split, base=None):
    base = base or DATA; P, Y = [], []
    for i, c in enumerate(classes):
        fs = sorted(glob.glob(f'{base}/{split}/{c}/*')); P += fs; Y += [i] * len(fs)
    return P, np.array(Y)
train_paths, y_train = files('train'); val_paths, y_val = files('val'); test_paths, y_test = files('test')

def hashes(ps): return np.stack([phash_bits(p) for p in tqdm(ps)])
Htr = hashes(train_paths)
def nearest(Hq):
    d, j = [], []
    for h in Hq: x = POP[Htr ^ h].sum(1); j.append(x.argmin()); d.append(x.min())
    return np.array(d), np.array(j)
d_te, j_te = nearest(hashes(test_paths)); d_va, _ = nearest(hashes(val_paths))
aud = pd.DataFrame({'split': ['test', 'val'], 'min': [d_te.min(), d_va.min()], 'p5': [np.percentile(d_te, 5), np.percentile(d_va, 5)],
                    'median': [np.median(d_te), np.median(d_va)], 'n_le_40bits': [(d_te <= 40).sum(), (d_va <= 40).sum()]})
aud.to_csv(f'{RES}/leakage_audit.csv', index=False); print(aud.to_string(index=False))

o = np.argsort(d_te)[:6]
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for k, i in enumerate(o):
    ax[0, k].imshow(Image.open(test_paths[i])); ax[0, k].set_title(f'test (dist {d_te[i]})', fontsize=8); ax[0, k].axis('off')
    ax[1, k].imshow(Image.open(train_paths[j_te[i]])); ax[1, k].set_title('nearest train', fontsize=8); ax[1, k].axis('off')
plt.tight_layout(); plt.savefig(f'{RES}/figures/leakage_closest_pairs.png', dpi=150); plt.show()

## 4. Shared pipeline: corruptions, data, metrics

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import ImageFilter, ImageEnhance
from sklearn.metrics import f1_score, recall_score, confusion_matrix, roc_auc_score
from scipy.optimize import minimize_scalar
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

CORR = {'blur_1': ('blur', 1), 'blur_2': ('blur', 2), 'blur_3': ('blur', 4),
        'dark_1': ('bright', .6), 'dark_2': ('bright', .4), 'dark_3': ('bright', .2),
        'bright_1': ('bright', 1.4), 'bright_2': ('bright', 1.8), 'bright_3': ('bright', 2.2),
        'jpeg_1': ('jpeg', 30), 'jpeg_2': ('jpeg', 15), 'jpeg_3': ('jpeg', 5)}
ALL_CONDS = ['clean'] + list(CORR)
FAMILIES = ['blur', 'dark', 'bright', 'jpeg']
def corrupt(img, kind, p):
    if kind == 'blur': return img.filter(ImageFilter.GaussianBlur(p))
    if kind == 'bright': return ImageEnhance.Brightness(img).enhance(p)
    b = io.BytesIO(); img.save(b, format='JPEG', quality=int(p)); b.seek(0); return Image.open(b).convert('RGB')

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]          # identical for all three models
norm = T.Normalize(MEAN, STD)
pre = T.Compose([T.Resize(IMGSZ), T.CenterCrop(IMGSZ)])
train_tf = T.Compose([T.RandomResizedCrop(IMGSZ, scale=(.6, 1.)), T.RandomHorizontalFlip(), T.ColorJitter(.2, .2, .1), T.ToTensor(), norm])
def tens(pil): return norm(T.functional.to_tensor(pil))

class DS(Dataset):
    def __init__(s, paths, y=None, cond='clean', train=False): s.paths, s.y, s.cond, s.train = paths, y, cond, train
    def __len__(s): return len(s.paths)
    def pil(s, i):
        img = pre(Image.open(s.paths[i]).convert('RGB'))
        return img if s.cond == 'clean' else corrupt(img, *CORR[s.cond])
    def __getitem__(s, i):
        x = train_tf(Image.open(s.paths[i]).convert('RGB')) if s.train else tens(s.pil(i))
        return x if s.y is None else (x, int(s.y[i]))

@torch.no_grad()
def get_logits(model, paths, cond='clean'):
    model.eval(); out = []
    for x in DataLoader(DS(paths, cond=cond), batch_size=64, num_workers=2):
        out.append(model(x.to(dev)).float().cpu().numpy())
    return np.concatenate(out)

def softmax(z, Tm=1.0):
    z = z / Tm; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def ece(p, y, bins=15):
    conf, ok = p.max(1), (p.argmax(1) == y).astype(float); edges = np.linspace(0, 1, bins + 1); o = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): o += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return o
def brier(p, y): return ((p - np.eye(p.shape[1])[y]) ** 2).sum(1).mean()
def nll(L, y, Tm): return -np.log(softmax(L, Tm)[np.arange(len(y)), y] + 1e-12).mean()
def risk_cov(p, y):
    o = np.argsort(-p.max(1)); ok = (p.argmax(1)[o] == y[o]).astype(float)
    n = np.arange(1, len(y) + 1); return n / len(y), np.cumsum(ok) / n
def pearson(a, b):
    a, b = a.ravel() - a.mean(), b.ravel() - b.mean(); return float((a * b).sum() / (np.sqrt((a ** 2).sum() * (b ** 2).sum()) + 1e-8))
def iou_top(a, b, q=.8):
    ta, tb = a >= np.quantile(a, q), b >= np.quantile(b, q); return float((ta & tb).sum() / max((ta | tb).sum(), 1))
def savefig(name): plt.tight_layout(); plt.savefig(f'{RES}/figures/{name}.png', dpi=200); plt.show()
print('classes:', classes, '| test images:', len(test_paths))

## 5. Build and train the three models
Same everything: ImageNet-pretrained backbone, full fine-tuning, AdamW + cosine schedule, same augmentation, **class-weighted cross-entropy** (the imbalance method; state it in the paper).
The best epoch is chosen by validation macro-F1.

In [ ]:
def build(name):
    if name == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, K)
    elif name == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1); m.classifier[1] = nn.Linear(m.classifier[1].in_features, K)
    elif name == 'mobilenet_v2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1); m.classifier[1] = nn.Linear(m.classifier[1].in_features, K)
    else: raise ValueError(name)
    return m.to(dev)
def cam_layer(name, m):                 # last convolutional block before the classification head
    if name == 'resnet50': return m.layer4[-1]
    return m.features[-1]               # efficientnet_b0 and mobilenet_v2

cw = torch.tensor(len(y_train) / (K * np.bincount(y_train)), dtype=torch.float32, device=dev)
print('class weights:', dict(zip(classes, cw.cpu().numpy().round(2))))

def train_model(name):
    g = torch.Generator().manual_seed(SEED)
    dl = DataLoader(DS(train_paths, y_train, train=True), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, generator=g)
    m = build(name); opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(dl)); scaler = torch.amp.GradScaler('cuda')
    lossf = nn.CrossEntropyLoss(weight=cw); best, hist = -1, []
    for ep in range(EPOCHS):
        m.train(); tl = 0
        for x, y in dl:
            x, y = x.to(dev), y.to(dev); opt.zero_grad()
            with torch.autocast('cuda', dtype=torch.float16): loss = lossf(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
        pv = get_logits(m, val_paths).argmax(1); f1 = f1_score(y_val, pv, average='macro')
        hist.append({'epoch': ep + 1, 'train_loss': tl / len(dl), 'val_macro_f1': f1}); print(name, hist[-1])
        if f1 > best: best = f1; torch.save(m.state_dict(), f'{OUT}/{name}.pt')
    pd.DataFrame(hist).to_csv(f'{RES}/train_history_{name}.csv', index=False)

for name in MODELS:
    if TRAIN or not os.path.exists(f'{OUT}/{name}.pt'): train_model(name)
    else: print(name, ': weights found in Drive, skipping training')

## 6. Run every model on clean + corrupted test images and corrupted validation images

In [ ]:
nets = {}
for name in MODELS:
    m = build(name); m.load_state_dict(torch.load(f'{OUT}/{name}.pt', map_location=dev)); m.eval(); nets[name] = m

VAL_CONDS = ['clean', 'blur_2', 'dark_2', 'bright_2', 'jpeg_2']     # used to fit the temperature
LG, LV = {}, {}
for name in MODELS:
    LG[name] = {c: get_logits(nets[name], test_paths, c) for c in tqdm(ALL_CONDS, desc=name + ' test')}
    LV[name] = {c: get_logits(nets[name], val_paths, c) for c in VAL_CONDS}
    np.savez(f'{RES}/logits_{name}.npz', y_test=y_test, y_val=y_val, **{'test_' + k: v for k, v in LG[name].items()}, **{'val_' + k: v for k, v in LV[name].items()})
print('done')

## 7. Baseline comparison (clean test set): macro-F1, per-class recall with 95% bootstrap CIs, confusion matrices

In [ ]:
def boot_ci(y, p, fn, B=1000):
    r = np.random.RandomState(SEED); n = len(y)
    v = [fn(y[i], p[i]) for i in (r.randint(0, n, n) for _ in range(B))]; return np.percentile(v, [2.5, 97.5])
rows = []
for name in MODELS:
    p = LG[name]['clean'].argmax(1); lo, hi = boot_ci(y_test, p, lambda a, b: f1_score(a, b, average='macro', zero_division=0))
    P = softmax(LG[name]['clean'])
    row = {'model': name, 'accuracy': (p == y_test).mean(), 'macro_f1': f1_score(y_test, p, average='macro'), 'macro_f1_CI': f'[{lo:.3f}, {hi:.3f}]',
           'ece': ece(P, y_test), 'brier': brier(P, y_test), 'n_errors': int((p != y_test).sum())}
    rc = recall_score(y_test, p, labels=list(range(K)), average=None, zero_division=0)
    for k, c in enumerate(classes):
        l, h = boot_ci(y_test, p, lambda a, b, k=k: recall_score(a, b, labels=[k], average=None, zero_division=0)[0])
        row[f'recall_{c}'] = f'{rc[k]:.3f} [{l:.2f},{h:.2f}]'
    rows.append(row)
base = pd.DataFrame(rows); base.to_csv(f'{RES}/baseline_clean.csv', index=False)
print(base.drop(columns=[c for c in base if c.startswith('recall_')]).round(4).to_string(index=False)); print(); print(base[['model'] + [c for c in base if c.startswith('recall_')]].to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a_, name in zip(ax, MODELS):
    cm = confusion_matrix(y_test, LG[name]['clean'].argmax(1), labels=range(K)); a_.imshow(cm, cmap='Blues'); a_.set_title(name)
    a_.set_xticks(range(K)); a_.set_xticklabels(classes, rotation=45, ha='right', fontsize=7); a_.set_yticks(range(K)); a_.set_yticklabels(classes, fontsize=7)
    for i in range(K):
        for j in range(K): a_.text(j, i, cm[i, j], ha='center', va='center', fontsize=7)
savefig('confusion_matrices_clean')

## 8. Robustness: all conditions, per-class vulnerability

In [ ]:
rows, rrows = [], []
for name in MODELS:
    for c in ALL_CONDS:
        P = softmax(LG[name][c]); pr = P.argmax(1)
        rows.append({'model': name, 'condition': c, 'accuracy': (pr == y_test).mean(), 'macro_f1': f1_score(y_test, pr, average='macro', zero_division=0),
                     'ece': ece(P, y_test), 'brier': brier(P, y_test), 'mean_conf': P.max(1).mean()})
        for k, r in enumerate(recall_score(y_test, pr, labels=list(range(K)), average=None, zero_division=0)):
            rrows.append({'model': name, 'condition': c, 'class': classes[k], 'recall': r})
met = pd.DataFrame(rows); rec = pd.DataFrame(rrows)
met.to_csv(f'{RES}/metrics_by_condition.csv', index=False); rec.to_csv(f'{RES}/per_class_recall.csv', index=False)
print(met.pivot(index='condition', columns='model', values='macro_f1').loc[ALL_CONDS].round(3).to_string())

fig, ax = plt.subplots(1, 4, figsize=(17, 3.6), sharey=True)
for a_, fam in zip(ax, FAMILIES):
    for name in MODELS:
        v = met[met.model == name].set_index('condition').macro_f1
        a_.plot(['clean', '1', '2', '3'], [v['clean']] + [v[f'{fam}_{i}'] for i in (1, 2, 3)], marker='o', label=name)
    a_.set_title(fam); a_.set_xlabel('severity'); a_.grid(alpha=.3)
ax[0].set_ylabel('macro-F1'); ax[0].legend(fontsize=7); savefig('robustness_macroF1')

vul = {}
for name in MODELS:
    pv = rec[rec.model == name].pivot(index='class', columns='condition', values='recall')
    vul[name] = pv['clean'] - pv[[c for c in pv if c.endswith('_3')]].mean(1)          # recall drop, mean over the 4 severity-3 corruptions
vul = pd.DataFrame(vul).loc[classes]; vul.to_csv(f'{RES}/class_vulnerability.csv'); print('\nRecall drop at severity 3 (clean minus mean of 4 corruptions):'); print(vul.round(3).to_string())

In [ ]:
# readable per-class vulnerability heatmap
fig, ax = plt.subplots(figsize=(6.5, 3.8)); im = ax.imshow(vul.values, cmap='Reds', vmin=0)
ax.set_xticks(range(len(MODELS))); ax.set_xticklabels(MODELS, rotation=20, ha='right'); ax.set_yticks(range(K)); ax.set_yticklabels(classes)
for i in range(K):
    for j in range(len(MODELS)): ax.text(j, i, f'{vul.values[i, j]:.2f}', ha='center', va='center')
ax.set_title('Per-class recall drop (severity 3)'); plt.colorbar(im); savefig('class_vulnerability')

# confusion matrices under the two hardest corruptions
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for r_, c in enumerate(['blur_3', 'jpeg_3']):
    for k_, name in enumerate(MODELS):
        cm = confusion_matrix(y_test, LG[name][c].argmax(1), labels=range(K)); a_ = ax[r_][k_]; a_.imshow(cm, cmap='Blues'); a_.set_title(f'{name} | {c}')
        a_.set_xticks(range(K)); a_.set_xticklabels(classes, rotation=45, ha='right', fontsize=7); a_.set_yticks(range(K)); a_.set_yticklabels(classes, fontsize=7)
        for i in range(K):
            for j in range(K): a_.text(j, i, cm[i, j], ha='center', va='center', fontsize=7, color='red' if cm[i, j] > cm.max() / 2 else 'black')
savefig('confusion_matrices_corrupted')

# do models collapse onto one class under corruption? (a majority class can look 'robust' only because of this)
for c in ['blur_3', 'jpeg_3']:
    tab = pd.DataFrame({m: np.bincount(LG[m][c].argmax(1), minlength=K) / len(y_test) for m in MODELS}, index=classes)
    tab['true_share'] = np.bincount(y_test, minlength=K) / len(y_test)
    print(f'\nShare of predictions per class under {c}:'); print(tab.round(3).to_string())

## 9. Calibration and selective prediction
A temperature fitted on a **clean** validation set can degenerate when the model is ~100% accurate there (the optimiser keeps sharpening). So we compare:
raw, `T_clean` (fitted on clean val) and `T_mixed` (fitted on clean + corrupted val). Test severity 3 is never used for fitting.

In [ ]:
Tt = {}
for name in MODELS:
    Lc, yc = LV[name]['clean'], y_val
    Lm = np.concatenate([LV[name][c] for c in VAL_CONDS]); ym = np.tile(y_val, len(VAL_CONDS))
    Tc = minimize_scalar(lambda t: nll(Lc, yc, t), bounds=(.5, 10), method='bounded').x
    Tm = minimize_scalar(lambda t: nll(Lm, ym, t), bounds=(.5, 10), method='bounded').x
    Tt[name] = (Tc, Tm); print(f'{name}: T_clean={Tc:.2f}  T_mixed={Tm:.2f}', '<- T_clean at a bound = degenerate fit (val set is ~perfect)' if (Tc < .55 or Tc > 9.9) else '')

crow = []
for name in MODELS:
    Tc, Tm = Tt[name]
    for c in ['clean', 'blur_3', 'dark_3', 'bright_3', 'jpeg_3']:
        L = LG[name][c]
        crow.append({'model': name, 'condition': c, 'ece_raw': ece(softmax(L), y_test), 'ece_Tclean': ece(softmax(L, Tc), y_test), 'ece_Tmixed': ece(softmax(L, Tm), y_test),
                     'brier_raw': brier(softmax(L), y_test), 'brier_Tmixed': brier(softmax(L, Tm), y_test)})
cal = pd.DataFrame(crow); cal.to_csv(f'{RES}/calibration.csv', index=False); print(cal.round(4).to_string(index=False))

# reliability diagram pooled over all 13 test conditions (otherwise a ~100%-accurate clean set shows nothing)
fig, ax = plt.subplots(1, 3, figsize=(15, 4)); edges = np.linspace(0, 1, 11)
for a_, name in zip(ax, MODELS):
    Lall = np.concatenate([LG[name][c] for c in ALL_CONDS]); yall = np.tile(y_test, len(ALL_CONDS))
    for lbl, P_ in [('raw', softmax(Lall)), ('T_mixed', softmax(Lall, Tt[name][1]))]:
        cf, ok = P_.max(1), P_.argmax(1) == yall; xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m_ = (cf > lo) & (cf <= hi)
            if m_.sum() >= 5: xs.append(cf[m_].mean()); ys.append(ok[m_].mean())
        a_.plot(xs, ys, marker='o', label=lbl)
    a_.plot([0, 1], [0, 1], 'k--'); a_.set_title(f'{name} (pooled corruptions)'); a_.set_xlabel('confidence'); a_.set_ylabel('accuracy'); a_.legend()
savefig('reliability_diagrams')

fig, ax = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True); srows = []
for a_, c in zip(ax, ['blur_3', 'jpeg_3', 'dark_3', 'bright_3']):
    for name in MODELS:
        cov, acc = risk_cov(softmax(LG[name][c]), y_test); a_.plot(cov, acc, label=name)
        srows.append({'model': name, 'condition': c, 'AURC': float((1 - acc).mean()),
                      **{f'acc@{int(l * 100)}%cov': acc[int(np.ceil(l * len(y_test))) - 1] for l in (1.0, .9, .8, .7, .5)}})
    a_.set_title(c); a_.set_xlabel('coverage'); a_.grid(alpha=.3)
ax[0].set_ylabel('accuracy on answered images'); ax[0].legend(fontsize=7); savefig('selective_prediction')
sel = pd.DataFrame(srows); sel.to_csv(f'{RES}/selective_prediction.csv', index=False); print(sel.round(4).to_string(index=False)); print('AURC: lower is better')

## 10. Grad-CAM: explanation stability under corruption and cross-model consistency
Same sample of test images for all three models. For each image, the heatmap of the **clean prediction's class** on the clean image is compared with
the heatmap of the same class on each corrupted version (Pearson correlation, SSIM, overlap of the top-20% pixels). 1 = identical heatmap.
Grad-CAM shows which regions drive the prediction. It does not prove they are disease lesions.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from skimage.metrics import structural_similarity as ssim

CAM_CONDS = ['blur_2', 'blur_3', 'dark_2', 'dark_3', 'bright_2', 'bright_3', 'jpeg_2', 'jpeg_3']
rng = np.random.RandomState(SEED)
cam_idx = rng.choice(len(test_paths), min(N_CAM, len(test_paths)), replace=False)
ds_img = DS(test_paths)
clean_cams, srows = {}, []
for name in MODELS:
    net = nets[name]; net.eval()
    for p in net.parameters(): p.requires_grad_(True)
    pred = LG[name]['clean'].argmax(1); store = []
    with GradCAM(model=net, target_layers=[cam_layer(name, net)]) as cam:
        for i in tqdm(cam_idx, desc=name):
            leaf = ds_img.pil(i)
            batch = torch.stack([tens(leaf)] + [tens(corrupt(leaf, *CORR[c])) for c in CAM_CONDS]).to(dev)
            Hm = cam(input_tensor=batch, targets=[ClassifierOutputTarget(int(pred[i]))] * len(batch))
            store.append(Hm[0])
            for c, h in zip(CAM_CONDS, Hm[1:]):
                srows.append({'model': name, 'idx': int(i), 'cond': c, 'correct': bool(pred[i] == y_test[i]), 'pearson': pearson(Hm[0], h),
                              'iou_top20': iou_top(Hm[0], h), 'ssim': float(ssim(Hm[0], h, data_range=1.0))})
    clean_cams[name] = np.stack(store)
stab = pd.DataFrame(srows); stab.to_csv(f'{RES}/gradcam_stability_per_image.csv', index=False)
ss = stab.groupby(['model', 'cond'])[['pearson', 'ssim', 'iou_top20']].mean().round(3); ss.to_csv(f'{RES}/gradcam_stability_summary.csv')
print(stab.pivot_table(index='cond', columns='model', values='pearson').loc[CAM_CONDS].round(3).to_string()); print('(mean Pearson between clean and corrupted heatmap)')

pv = stab.pivot_table(index='cond', columns='model', values='pearson').loc[CAM_CONDS]
fig, ax = plt.subplots(figsize=(8, 3.8)); w = .25
for k, name in enumerate(MODELS): ax.bar(np.arange(len(CAM_CONDS)) + k * w, pv[name].values, w, label=name)
ax.set_xticks(np.arange(len(CAM_CONDS)) + w); ax.set_xticklabels(CAM_CONDS, rotation=45); ax.set_ylabel('Grad-CAM Pearson (clean vs corrupted)'); ax.legend(fontsize=7); savefig('gradcam_stability')

# cross-model consistency on leaves that all three models classify correctly
okall = np.array([all(LG[m]['clean'].argmax(1)[i] == y_test[i] for m in MODELS) for i in cam_idx])
pairs = [(a, b) for a in range(3) for b in range(a + 1, 3)]
cm_rows = [{'pair': f'{MODELS[a]} vs {MODELS[b]}', 'mean_pearson': np.mean([pearson(clean_cams[MODELS[a]][j], clean_cams[MODELS[b]][j]) for j in np.where(okall)[0]]),
            'mean_iou_top20': np.mean([iou_top(clean_cams[MODELS[a]][j], clean_cams[MODELS[b]][j]) for j in np.where(okall)[0]])} for a, b in pairs]
cmd = pd.DataFrame(cm_rows); cmd.to_csv(f'{RES}/cross_model_consistency.csv', index=False); print('\nCross-model heatmap agreement (n =', int(okall.sum()), 'leaves):'); print(cmd.round(3).to_string(index=False))

pick, seen = [], set()                       # one leaf per class, then fill up to 6
for j in np.where(okall)[0]:
    cl = y_test[cam_idx[j]]
    if cl not in seen: seen.add(cl); pick.append(j)
pick = (pick + [j for j in np.where(okall)[0] if j not in pick])[:6]
fig, ax = plt.subplots(len(pick), 4, figsize=(9, 2.3 * len(pick)), squeeze=False)
for r_, j in enumerate(pick):
    im = np.asarray(ds_img.pil(cam_idx[j])).astype(np.float32) / 255
    ax[r_][0].imshow(im); ax[r_][0].set_ylabel(classes[y_test[cam_idx[j]]], fontsize=7)
    for c_, name in enumerate(MODELS): ax[r_][c_ + 1].imshow(show_cam_on_image(im, clean_cams[name][j], use_rgb=True))
    for a_ in ax[r_]: a_.set_xticks([]); a_.set_yticks([])
for c_, t in enumerate(['original'] + MODELS): ax[0][c_].set_title(t, fontsize=8)
savefig('cross_model_gradcam')

# example: heatmap change under corruption (one leaf, each model)
j = pick[0]; i = cam_idx[j]; leaf = ds_img.pil(i); conds = ['clean', 'blur_2', 'blur_3', 'jpeg_3', 'dark_3']
fig, ax = plt.subplots(len(MODELS), len(conds), figsize=(2.2 * len(conds), 2.3 * len(MODELS)), squeeze=False)
for r_, name in enumerate(MODELS):
    with GradCAM(model=nets[name], target_layers=[cam_layer(name, nets[name])]) as cam:
        for c_, cd in enumerate(conds):
            im = leaf if cd == 'clean' else corrupt(leaf, *CORR[cd])
            h = cam(input_tensor=tens(im)[None].to(dev), targets=[ClassifierOutputTarget(int(LG[name]['clean'].argmax(1)[i]))])[0]
            ax[r_][c_].imshow(show_cam_on_image(np.asarray(im).astype(np.float32) / 255, h, use_rgb=True)); ax[r_][c_].axis('off')
            if r_ == 0: ax[r_][c_].set_title(cd, fontsize=8)
    ax[r_][0].text(0, -8, name, fontsize=7)
savefig('gradcam_under_corruption')

## 11. Idea C: can explanation instability flag wrong predictions?
On clean data the models make almost no errors, so errors are studied on corrupted images (blur, JPEG), where they are common.
To be **deployable** (no clean reference at test time), instability is a self-consistency score: the Grad-CAM of the image versus the Grad-CAM of its horizontally flipped copy (flipped back), for the model's predicted class.
Signals compared by AUROC for detecting wrong predictions: low confidence, unstable explanation, and the rank-average of both. 0.5 = useless, 1.0 = perfect.
If combined is not better than confidence alone, report that honestly. It is still a valid finding.

In [ ]:
ic_idx = np.random.RandomState(SEED + 1).choice(len(test_paths), min(N_IDEA_C, len(test_paths)), replace=False)
rk = lambda v: v.argsort().argsort() / len(v)
def acc_at(score, e, cov=.8): keep = np.argsort(score)[:int(cov * len(e))]; return 1 - e[keep].mean()      # reject the highest-risk images
icr = []
for name in MODELS:
    net = nets[name]
    with GradCAM(model=net, target_layers=[cam_layer(name, net)]) as cam:
        for c in ['blur_2', 'blur_3', 'jpeg_2', 'jpeg_3']:
            P = softmax(LG[name][c]); rec_ = []
            for i in tqdm(ic_idx, desc=f'{name} {c}'):
                t = tens(corrupt(ds_img.pil(i), *CORR[c])); pc = int(P[i].argmax())
                Hm = cam(input_tensor=torch.stack([t, t.flip(-1)]).to(dev), targets=[ClassifierOutputTarget(pc)] * 2)
                rec_.append((1 - P[i].max(), 1 - pearson(Hm[0], Hm[1][:, ::-1]), int(pc != y_test[i])))
            a, b, e = map(np.array, zip(*rec_))
            if 10 <= e.sum() < len(e):
                comb = rk(a) + rk(b)
                icr.append({'model': name, 'condition': c, 'n': len(e), 'n_errors': int(e.sum()), 'auroc_low_conf': roc_auc_score(e, a), 'auroc_unstable_cam': roc_auc_score(e, b),
                            'auroc_combined': roc_auc_score(e, comb), 'acc@80%cov_conf': acc_at(a, e), 'acc@80%cov_combined': acc_at(comb, e)})
            else: icr.append({'model': name, 'condition': c, 'n': len(e), 'n_errors': int(e.sum())})
icdf = pd.DataFrame(icr); icdf.to_csv(f'{RES}/idea_c_error_detection.csv', index=False); print(icdf.round(3).to_string(index=False))
print('Note: AUROCs from few errors are noisy; always report n_errors.')

## 12. (Optional) external test set
Put your own photos in `EXT_DIR/<class name>/*.jpg` using the same class folder names, set `EXT_DIR` in the settings cell, and rerun this cell.
Even 50-100 images from a different source make the generalisation claim much stronger.

In [ ]:
if EXT_DIR and os.path.isdir(EXT_DIR):
    ext_p, ext_y = [], []
    for i, c in enumerate(classes):
        fs = [f for f in sorted(glob.glob(f'{EXT_DIR}/{c}/*')) if os.path.splitext(f)[1].lower() in IMGEXT]; ext_p += fs; ext_y += [i] * len(fs)
    ext_y = np.array(ext_y); er = []
    for name in MODELS:
        P = softmax(get_logits(nets[name], ext_p)); pr = P.argmax(1)
        er.append({'model': name, 'n': len(ext_y), 'accuracy': (pr == ext_y).mean(), 'macro_f1': f1_score(ext_y, pr, average='macro', zero_division=0), 'ece': ece(P, ext_y)})
    er = pd.DataFrame(er); er.to_csv(f'{RES}/external_test.csv', index=False); print(er.round(4).to_string(index=False))
else: print('EXT_DIR not set: skipped. In the paper, state that generalisation to unseen image sources was not tested.')

## 13. Key numbers for the paper

In [ ]:
print('=== KEY RESULTS ===')
print(base.drop(columns=[c for c in base if c.startswith('recall_')]).round(4).to_string(index=False))
w = met[met.condition != 'clean'].groupby('model').macro_f1.mean().round(3)
print('\nMean macro-F1 over all 12 corruptions:', w.to_dict())
print('Most robust model:', w.idxmax(), '| most vulnerable class (mean over models):', vul.mean(1).idxmax())
print('Most stable Grad-CAM (mean Pearson, all corruptions):', stab.groupby('model').pearson.mean().round(3).to_dict())
print('\nAll CSVs and figures saved in:', RES)
print('Reminder: state the imbalance method (class-weighted CE), the dedup threshold + count removed, and that results come from one Kaggle dataset and one seed.')